In [1]:
import sys
import subprocess
from pathlib import Path

PROJECT_DIR = Path.cwd()
requirements_file = PROJECT_DIR / "requirements.txt"
if not requirements_file.exists():
    raise FileNotFoundError("Open resume_analyzer.ipynb from the AI_Resume_Analyzer_Project folder.")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_file)])

0

In [2]:
# Setup 2: Import packages and set project paths
import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_DIR = Path.cwd()
RESUME_DIR = PROJECT_DIR / "resumes"
PROMPT_DIR = PROJECT_DIR / "prompts"
OUTPUT_DIR = PROJECT_DIR / "outputs"
LOG_DIR = PROJECT_DIR / "logs"
DATA_DIR = PROJECT_DIR / "data"
for folder in [RESUME_DIR, PROMPT_DIR, OUTPUT_DIR, LOG_DIR, DATA_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_DIR / ".env")
print("Project folder:", PROJECT_DIR)
print("Python:", sys.version.split()[0])
print("Gemini model:", os.getenv("GEMINI_MODEL", "gemini-2.5-flash"))
print("API key configured:", bool(os.getenv("GEMINI_API_KEY")))  # Never print the key

Project folder: d:\New folder\AI_Resume_Analyzer_Project\AI_Resume_Analyzer_Project
Python: 3.12.10
Gemini model: gemini-2.5-flash
API key configured: False


In [3]:
# Setup 3: Confirm the helper files and sample data are present
required_paths = [
    PROJECT_DIR / "resume_analyzer_core.py",
    PROJECT_DIR / "app.py",
    PROJECT_DIR / "tests" / "test_app.py",
    PROJECT_DIR / "data" / "golden_set.json",
]
missing = [str(path.name) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"Project bundle is incomplete. Missing: {missing}")

import importlib
import resume_analyzer_core as core
importlib.reload(core)
print("Project helper files found. Core module loaded.")

c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Project helper files found. Core module loaded.


In [ ]:
# Setup 4: Gemini API key check 

load_dotenv(PROJECT_DIR / ".env", override=True)
GEMINI_API_KEY_PRESENT = bool(os.getenv("GEMINI_API_KEY", "").strip())
print("GEMINI_API_KEY present:", GEMINI_API_KEY_PRESENT)
if not GEMINI_API_KEY_PRESENT:
    print("Next action: copy .env.example to .env and add GEMINI_API_KEY=your_key.")
    print("You can still complete the offline PDF/TXT reading cells before making API calls.")
else:
    print("Key detected securely. The key value is not displayed.")

GEMINI_API_KEY present: False
Next action: copy .env.example to .env and add GEMINI_API_KEY=your_key.
You can still complete the offline PDF/TXT reading cells before making API calls.


In [5]:
# P1.1: Create or switch to the required Git branch
# This creates a local Git repository if the project folder is not already under Git.

def run_git(args):
    return subprocess.run(["git", *args], cwd=PROJECT_DIR, capture_output=True, text=True)

git_check = run_git(["rev-parse", "--is-inside-work-tree"])
if git_check.returncode != 0:
    run_git(["init"])
    print("Initialized a local Git repository.")

branch_result = run_git(["branch", "--show-current"])
current_branch = branch_result.stdout.strip()
branch_list = run_git(["branch", "--list", "feature/resume-analyzer"])
if current_branch != "feature/resume-analyzer":
    if branch_list.stdout.strip():
        switch_result = run_git(["switch", "feature/resume-analyzer"])
    else:
        switch_result = run_git(["switch", "-c", "feature/resume-analyzer"])
    if switch_result.returncode != 0:
        print(switch_result.stderr)
        raise RuntimeError("Could not switch to feature/resume-analyzer. Check your Git installation.")
print("Current Git branch:", run_git(["branch", "--show-current"]).stdout.strip())

Initialized a local Git repository.
Current Git branch: feature/resume-analyzer


In [6]:
# P1.2: List the eight supplied dummy resumes
resume_files = sorted([
    path for path in RESUME_DIR.iterdir()
    if path.suffix.lower() in {".pdf", ".txt"} and not path.name.endswith("_test.txt") and not path.name.startswith("scanned_")
])
print("Resume count:", len(resume_files))
for path in resume_files:
    print("-", path.name)
assert len(resume_files) == 8, "The assessment requires eight sample resumes."

Resume count: 8
- resume_01.pdf
- resume_02.pdf
- resume_03.pdf
- resume_04.pdf
- resume_05.txt
- resume_06.txt
- resume_07.txt
- resume_08.txt


In [7]:
# P1.3–P1.4: Read and clean each resume; demonstrate the scanned-PDF message
resume_texts = {}
read_errors = {}
for path in resume_files:
    try:
        resume_texts[path.name] = core.read_resume(path)
    except Exception as exc:
        read_errors[path.name] = f"{type(exc).__name__}: {exc}"

print("Successfully read:", len(resume_texts))
print("Read errors:", read_errors)
print("\nPreview of resume_01:")
print(resume_texts.get("resume_01.pdf", "Not available")[:700])

scanned_path = RESUME_DIR / "scanned_resume_test.pdf"
try:
    core.read_resume(scanned_path)
except core.ScannedPDFError as exc:
    print("Scanned PDF test message:", exc)

Successfully read: 8
Read errors: {}

Preview of resume_01:
sofia.reed@example.com | +1-555-0101 | Fictional location: Example City
SUMMARY
Python developer with 3.5 years of experience building backend services and REST APIs.
TECHNICAL SKILLS
Python, FastAPI, SQL, Git, pytest, Docker
EXPERIENCE
Backend Developer | Example Software Studio | 2022 - Present
- Built REST APIs using Python and FastAPI for internal business applications.
- Wrote automated tests with pytest and reviewed code using Git.
- Designed SQL queries and worked with PostgreSQL databases.
- Packaged local development services with Docker.
EDUCATION
Bachelor of Science in Information Technology | Example University | 2021
ADDITIONAL INFORMATION
Interested in reliable backend systems 
Scanned PDF test message: PDF has no selectable text. It may be scanned/image-only; OCR is not included in this project.


In [ ]:
# P1.5: Report character counts and exact Gemini tokenizer counts
# Count Tokens uses the Gemini model tokenizer. It is a count operation, not a content-generation call.
if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError("Add your key to .env before running this token-count cell.")

client = core.get_gemini_client()
count_rows = []
for filename, text in resume_texts.items():
    token_count = core.count_gemini_tokens(text, client=client)
    count_rows.append({
        "resume": filename,
        "characters": len(text),
        "tokens": token_count,
    })
    time.sleep(6.0)
resume_counts_df = pd.DataFrame(count_rows).sort_values("resume")
display(resume_counts_df)
resume_counts_df.to_csv(OUTPUT_DIR / "resume_character_token_counts.csv", index=False)
print("Saved resume_character_token_counts.csv")

In [9]:
# P1.4: Test empty and scanned files; both must fail with clear messages
validation_fixtures = [RESUME_DIR / "empty_resume_test.txt", scanned_path]
for fixture in validation_fixtures:
    try:
        core.read_resume(fixture)
    except core.ResumeReadError as exc:
        print(f"{fixture.name} -> {type(exc).__name__}: {exc}")

empty_resume_test.txt -> EmptyResumeError: No readable text was found in the resume.
scanned_resume_test.pdf -> ScannedPDFError: PDF has no selectable text. It may be scanned/image-only; OCR is not included in this project.


In [10]:
# P1.6: Display the Pydantic JSON schema used for structured extraction
from resume_analyzer_core import ResumeData, JobMatch, JobMatchBatch

resume_schema = ResumeData.model_json_schema()
print(json.dumps(resume_schema, indent=2))
(OUTPUT_DIR / "resume_schema.json").write_text(json.dumps(resume_schema, indent=2), encoding="utf-8")

{
  "additionalProperties": false,
  "description": "Required candidate extraction schema from the assessment.",
  "properties": {
    "candidate_name": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "title": "Candidate Name"
    },
    "skills": {
      "items": {
        "type": "string"
      },
      "minItems": 1,
      "title": "Skills",
      "type": "array"
    },
    "years_of_experience": {
      "anyOf": [
        {
          "maximum": 50,
          "minimum": 0,
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "title": "Years Of Experience"
    },
    "education": {
      "items": {
        "type": "string"
      },
      "title": "Education",
      "type": "array"
    },
    "strengths": {
      "items": {
        "type": "string"
      },
      "title": "Strengths",
      "type": "array"
    },
    "gaps": {
      "items": {
        "type": "st

1245

In [11]:
# P1.7: Load and version the extraction system prompt (v1)
EXTRACTION_PROMPT_V1 = (PROMPT_DIR / "extraction_prompt_v1.txt").read_text(encoding="utf-8")
print(EXTRACTION_PROMPT_V1)
print("Prompt saved at:", PROMPT_DIR / "extraction_prompt_v1.txt")

ROLE: You are a careful resume information extraction assistant.

TASK: Read the supplied resume text and extract only facts explicitly supported by the text.

OUTPUT FORMAT: Return one JSON object that conforms to the response schema supplied separately by the API.

CONSTRAINTS:
- Never invent, infer, or guess candidate facts.
- Use null for missing scalar values such as candidate_name or years_of_experience.
- Use an empty list when no evidence exists for education, strengths, or gaps.
- The skills list must not be empty. If no skills are explicitly stated, use ["Not provided"] as a transparent placeholder.
- Do not infer age, gender, ethnicity, religion, health, disability, family status, or other protected/sensitive traits.
- Do not treat contact details as skills or qualifications.
- Preserve skill names clearly; remove duplicate skills.
- Years of experience must be a number from 0 to 50 only when the resume gives enough evidence; otherwise null.
- Do not use outside knowledge.



In [ ]:
# P1.8: Run one resume first and inspect the structured JSON result
first_resume_name = resume_files[0].name
first_result = core.extract_resume_with_retry(
    resume_texts[first_resume_name],
    prompt_template=EXTRACTION_PROMPT_V1,
    client=client,
    max_attempts=3,
    pause_seconds=1.0,
)
print("Success:", first_result["success"])
print("Attempts:", first_result["attempts"])
print("Usage:", first_result["usage"])
if first_result["success"]:
    print(json.dumps(first_result["data"], indent=2))
else:
    print("Error:", first_result["error"])

In [ ]:
# P1.8: Run JSON-mode extraction for all eight resumes at temperature 0
# Each resume is sent separately so results and usage can be tracked individually.
extraction_results = {}
for index, (filename, text) in enumerate(resume_texts.items(), start=1):
    if filename == first_resume_name and first_result["success"]:
        result = first_result
    else:
        result = core.extract_resume_with_retry(
            text, prompt_template=EXTRACTION_PROMPT_V1, client=client,
            max_attempts=3, pause_seconds=3.0,
        )
    extraction_results[filename] = result
    print(f"{index}/8 {filename}: success={result['success']}, attempts={result['attempts']}, tokens={result['usage']['total_tokens']}")
    time.sleep(6.0)

successful_extractions = {name: result["data"] for name, result in extraction_results.items() if result["success"]}
(OUTPUT_DIR / "extraction_results_v1.json").write_text(json.dumps(successful_extractions, indent=2), encoding="utf-8")
print("Successful extractions:", len(successful_extractions), "/ 8")
if len(successful_extractions) != 8:
    raise RuntimeError("Not all eight resumes were extracted. Review errors, wait if rate-limited, then rerun only failed resumes before continuing.")

In [ ]:
# P1.9: Prepare a manual review table for three resumes
# Compare each extracted field with the source text and the golden set. Mark errors only after manual inspection.
golden_set = json.loads((DATA_DIR / "golden_set.json").read_text(encoding="utf-8"))
golden_by_file = {item["file"]: item for item in golden_set}
review_rows = []
for filename in list(resume_texts.keys())[:3]:
    prediction = successful_extractions.get(filename, {})
    gold = golden_by_file[filename]
    review_rows.append({
        "resume": filename,
        "expected_name": gold["candidate_name"],
        "predicted_name": prediction.get("candidate_name"),
        "expected_skills": ", ".join(gold["skills"]),
        "predicted_skills": ", ".join(prediction.get("skills", [])),
        "manual_error_count": "FILL_AFTER_REVIEW",
        "review_notes": "FILL_AFTER_REVIEW",
    })
manual_review_df = pd.DataFrame(review_rows)
display(manual_review_df)
manual_review_df.to_csv(OUTPUT_DIR / "manual_review_3_resumes.csv", index=False)
print("Open outputs/manual_review_3_resumes.csv and replace FILL_AFTER_REVIEW after comparing each result with its source.")

In [15]:
# P1.9: Enter the total number of errors found during your manual review
# Change this value after you inspect the three resumes. Do not leave it guessed.
manual_error_count = None  # Example: 2. Replace None with your real count after reviewing.
if manual_error_count is None:
    print("Manual review not recorded yet. Inspect the 3 rows and set manual_error_count.")
else:
    manual_review_summary = {"resumes_reviewed": 3, "errors_found": int(manual_error_count)}
    (OUTPUT_DIR / "manual_review_summary.json").write_text(json.dumps(manual_review_summary, indent=2), encoding="utf-8")
    print(manual_review_summary)

Manual review not recorded yet. Inspect the 3 rows and set manual_error_count.


In [16]:
# P1.10: Version the prompt and prepare the Git commit
# Run the Git commands in the VS Code terminal after reviewing the prompt and outputs.
print("Prompt version:", PROMPT_DIR / "extraction_prompt_v1.txt")
print("Suggested terminal commands:")
print('git add prompts/extraction_prompt_v1.txt outputs/resume_schema.json outputs/manual_review_3_resumes.csv')
print('git commit -m "Add resume extraction schema and prompt v1"')

Prompt version: d:\New folder\AI_Resume_Analyzer_Project\AI_Resume_Analyzer_Project\prompts\extraction_prompt_v1.txt
Suggested terminal commands:
git add prompts/extraction_prompt_v1.txt outputs/resume_schema.json outputs/manual_review_3_resumes.csv
git commit -m "Add resume extraction schema and prompt v1"


In [17]:
# P2.1: Inspect the validation rules on ResumeData
print(ResumeData.model_json_schema())
valid_example = {
    "candidate_name": "Example Candidate",
    "skills": ["Python"],
    "years_of_experience": 2.0,
    "education": [],
    "strengths": [],
    "gaps": [],
}
print("Valid example:", ResumeData.model_validate(valid_example))

{'additionalProperties': False, 'description': 'Required candidate extraction schema from the assessment.', 'properties': {'candidate_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'title': 'Candidate Name'}, 'skills': {'items': {'type': 'string'}, 'minItems': 1, 'title': 'Skills', 'type': 'array'}, 'years_of_experience': {'anyOf': [{'maximum': 50, 'minimum': 0, 'type': 'number'}, {'type': 'null'}], 'title': 'Years Of Experience'}, 'education': {'items': {'type': 'string'}, 'title': 'Education', 'type': 'array'}, 'strengths': {'items': {'type': 'string'}, 'title': 'Strengths', 'type': 'array'}, 'gaps': {'items': {'type': 'string'}, 'title': 'Gaps', 'type': 'array'}}, 'required': ['candidate_name', 'skills', 'years_of_experience', 'education', 'strengths', 'gaps'], 'title': 'ResumeData', 'type': 'object'}
Valid example: candidate_name='Example Candidate' skills=['Python'] years_of_experience=2.0 education=[] strengths=[] gaps=[]


In [18]:
# P2.2: Demonstrate a validation error for years_of_experience > 50
invalid_example = {**valid_example, "years_of_experience": 60}
try:
    ResumeData.model_validate(invalid_example)
except Exception as exc:
    print("Expected validation error:")
    print(exc)

Expected validation error:
1 validation error for ResumeData
years_of_experience
  Input should be less than or equal to 50 [type=less_than_equal, input_value=60, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [19]:
# P2.2–P2.3: Demonstrate retry feedback without making a live API call
# First response is invalid (61 years); the second response fixes it (6 years).
mock_outputs = iter([
    (json.dumps({**valid_example, "years_of_experience": 61}), {"input_tokens": 20, "output_tokens": 15, "total_tokens": 35}),
    (json.dumps({**valid_example, "years_of_experience": 6}), {"input_tokens": 25, "output_tokens": 15, "total_tokens": 40}),
])
def mock_generator(prompt, response_model):
    return next(mock_outputs)

retry_demo = core.validate_json_with_retry(
    "Extract the candidate resume.", ResumeData,
    generate_fn=mock_generator, max_attempts=3, pause_seconds=0,
    log_path=LOG_DIR / "retry_demo.jsonl", stage="retry_demo",
)
print("Retry result:", retry_demo)
assert retry_demo["success"] and retry_demo["attempts"] == 2

Retry result: {'success': True, 'data': {'candidate_name': 'Example Candidate', 'skills': ['Python'], 'years_of_experience': 6.0, 'education': [], 'strengths': [], 'gaps': []}, 'usage': {'input_tokens': 45, 'output_tokens': 30, 'total_tokens': 75}, 'attempts': 2, 'error': None}


In [20]:
# P2.3: Demonstrate the clear failure object after three invalid attempts
fail_outputs = iter([
    ('{"candidate_name":"X","skills":[],"years_of_experience":70,"education":[],"strengths":[],"gaps":[]}', {"input_tokens": 5, "output_tokens": 5, "total_tokens": 10}),
    ('not json', {"input_tokens": 5, "output_tokens": 5, "total_tokens": 10}),
    ('{"candidate_name":"X","skills":[],"years_of_experience":70,"education":[],"strengths":[],"gaps":[]}', {"input_tokens": 5, "output_tokens": 5, "total_tokens": 10}),
])
def always_bad_generator(prompt, response_model):
    return next(fail_outputs)

failed_demo = core.validate_json_with_retry(
    "Test a failed extraction.", ResumeData, generate_fn=always_bad_generator,
    max_attempts=3, pause_seconds=0, log_path=LOG_DIR / "retry_failure_demo.jsonl",
)
print(failed_demo)
assert failed_demo["success"] is False and failed_demo["attempts"] == 3

{'success': False, 'data': None, 'usage': {'input_tokens': 15, 'output_tokens': 15, 'total_tokens': 30}, 'attempts': 3, 'error': '2 validation errors for ResumeData\nskills\n  List should have at least 1 item after validation, not 0 [type=too_short, input_value=[], input_type=list]\n    For further information visit https://errors.pydantic.dev/2.14/v/too_short\nyears_of_experience\n  Input should be less than or equal to 50 [type=less_than_equal, input_value=70, input_type=int]\n    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal'}


In [21]:
# P2.4: Test one messy resume and one empty resume
messy_text = """  
  PRIYA RAMAN   

  SQL   |   Excel  

  Page 1 of 1  
 """
print("Messy text after cleaning:")
print(core.clean_resume_text(messy_text))

empty_path = RESUME_DIR / "empty_resume_test.txt"
try:
    core.read_resume(empty_path)
except core.EmptyResumeError as exc:
    print("Empty resume test:", exc)

Messy text after cleaning:
PRIYA RAMAN
SQL | Excel
Empty resume test: No readable text was found in the resume.


In [22]:
# P2.5: Review metadata-only retry logs and actual extraction token usage
log_path = LOG_DIR / "api_attempts.jsonl"
if log_path.exists():
    log_records = [json.loads(line) for line in log_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    logs_df = pd.DataFrame(log_records)
    display(logs_df.tail(12))
else:
    print("No live API attempt logs yet. Run the extraction cells first.")

# The log records contain attempt metadata and token usage only; resume content and API keys are not written.

No live API attempt logs yet. Run the extraction cells first.


In [23]:
# P2.6: Create two fictional job descriptions
jobs = [
    {
        "title": "Python Developer",
        "description": "Job Title: Python Developer\nRequired skills: Python, FastAPI, SQL, REST APIs, Git, automated testing.\nResponsibilities: Build backend APIs, connect databases, write tests, review code, and document services."
    },
    {
        "title": "Data Analyst",
        "description": "Job Title: Data Analyst\nRequired skills: SQL, Python, pandas, Excel, Power BI, data cleaning, reporting.\nResponsibilities: Clean business data, write SQL queries, prepare dashboards, and explain findings to stakeholders."
    },
]
(OUTPUT_DIR / "job_descriptions.json").write_text(json.dumps(jobs, indent=2), encoding="utf-8")
for job in jobs:
    print(job["title"], "->", job["description"])

Python Developer -> Job Title: Python Developer
Required skills: Python, FastAPI, SQL, REST APIs, Git, automated testing.
Responsibilities: Build backend APIs, connect databases, write tests, review code, and document services.
Data Analyst -> Job Title: Data Analyst
Required skills: SQL, Python, pandas, Excel, Power BI, data cleaning, reporting.
Responsibilities: Clean business data, write SQL queries, prepare dashboards, and explain findings to stakeholders.


In [ ]:
# P2.7: Load and version the matching prompt
MATCHING_PROMPT_V1 = (PROMPT_DIR / "matching_prompt_v1.txt").read_text(encoding="utf-8")
print(MATCHING_PROMPT_V1)

In [ ]:
# P2.8: Run all eight resumes against both job descriptions
# One JSON response per resume contains a match result for each of the two jobs.
matching_results_v1 = {}
for index, (filename, candidate_data) in enumerate(successful_extractions.items(), start=1):
    result = core.match_resume_to_jobs_with_retry(
        candidate_data, jobs, prompt_template=MATCHING_PROMPT_V1,
        client=client, max_attempts=3, pause_seconds=3.0,
    )
    matching_results_v1[filename] = result
    titles = [m["job_title"] for m in result["data"]["matches"]] if result["success"] else []
    print(f"{index}/8 {filename}: success={result['success']}, jobs={titles}, tokens={result['usage']['total_tokens']}")
    time.sleep(6.0)
(OUTPUT_DIR / "matching_results_v1.json").write_text(json.dumps(matching_results_v1, indent=2), encoding="utf-8")
if len([r for r in matching_results_v1.values() if r["success"]]) != 8:
    raise RuntimeError("Matching did not succeed for all eight resumes. Review errors and rerun failed candidates.")

In [ ]:
# P2.9: Run the same resume/job pairs a second time to check score consistency
matching_results_v2 = {}
for index, (filename, candidate_data) in enumerate(successful_extractions.items(), start=1):
    result = core.match_resume_to_jobs_with_retry(
        candidate_data, jobs, prompt_template=MATCHING_PROMPT_V1,
        client=client, max_attempts=3, pause_seconds=3.0,
    )
    matching_results_v2[filename] = result
    print(f"Repeat {index}/8 {filename}: success={result['success']}")
    time.sleep(6.0)
(OUTPUT_DIR / "matching_results_repeat.json").write_text(json.dumps(matching_results_v2, indent=2), encoding="utf-8")
if len([r for r in matching_results_v2.values() if r["success"]]) != 8:
    raise RuntimeError("The second consistency run is incomplete. Fix failures before calculating consistency.")

In [25]:
# P2.8–P2.9: Build ranking table and calculate average score difference
ranking_rows = []
for filename, first in matching_results_v1.items():
    second = matching_results_v2.get(filename, {})
    if not first.get("success") or not second.get("success"):
        continue
    second_by_title = {m["job_title"]: m for m in second["data"]["matches"]}
    for match in first["data"]["matches"]:
        repeated = second_by_title.get(match["job_title"], {})
        repeat_score = repeated.get("score")
        ranking_rows.append({
            "resume": filename,
            "candidate_name": successful_extractions[filename].get("candidate_name"),
            "job_title": match["job_title"],
            "score_run_1": match["score"],
            "score_run_2": repeat_score,
            "absolute_score_difference": abs(match["score"] - repeat_score) if repeat_score is not None else None,
            "matched_skills": ", ".join(match["matched_skills"]),
            "missing_skills": ", ".join(match["missing_skills"]),
            "reasoning": match["reasoning"],
        })
ranking_df = pd.DataFrame(ranking_rows)
if not ranking_df.empty:
    ranking_df = ranking_df.sort_values(["job_title", "score_run_1"], ascending=[True, False])
    display(ranking_df)
    print("Average absolute score difference:", ranking_df["absolute_score_difference"].mean())
    ranking_df.to_csv(OUTPUT_DIR / "candidate_rankings.csv", index=False)
else:
    print("No successful matching results yet. Check the Gemini quota and previous cell output.")

No successful matching results yet. Check the Gemini quota and previous cell output.


In [26]:
# P2.10: Four-line fairness and human-review note for your submission
fairness_note = """Automated resume scores can reproduce bias in historical hiring data and may reward wording rather than real ability.
A model can miss transferable skills or misunderstand non-traditional experience, career breaks, and different writing styles.
The analyzer is instructed to use only job-related evidence and ignore protected or sensitive personal attributes, but this does not guarantee fairness.
A human reviewer must verify source evidence, allow candidates to correct errors, and never use the score as the sole hiring decision."""
print(fairness_note)
(OUTPUT_DIR / "fairness_note.txt").write_text(fairness_note, encoding="utf-8")

Automated resume scores can reproduce bias in historical hiring data and may reward wording rather than real ability.
A model can miss transferable skills or misunderstand non-traditional experience, career breaks, and different writing styles.
The analyzer is instructed to use only job-related evidence and ignore protected or sensitive personal attributes, but this does not guarantee fairness.
A human reviewer must verify source evidence, allow candidates to correct errors, and never use the score as the sole hiring decision.


532

In [ ]:
# Optional Part 2 bonus: Compare LLM scores with a deterministic keyword-match baseline
import re
def keyword_match_score(skills, job_description):
    job_terms = set(re.findall(r"[a-zA-Z][a-zA-Z+#.-]+", job_description.lower()))
    skill_terms = [skill for skill in skills if skill.lower() in job_terms]
    required_terms = [term for term in ["python", "fastapi", "sql", "rest", "testing", "pandas", "excel", "power", "reporting"] if term in job_terms]
    matched = sum(1 for term in required_terms if any(term in skill.lower() for skill in skills))
    return round(100 * matched / len(required_terms), 1) if required_terms else 0.0

bonus_rows = []
for filename, data in successful_extractions.items():
    for job in jobs:
        bonus_rows.append({"resume": filename, "job_title": job["title"], "keyword_score": keyword_match_score(data["skills"], job["description"])})
bonus_keyword_df = pd.DataFrame(bonus_rows)
display(bonus_keyword_df.head(10))
bonus_keyword_df.to_csv(OUTPUT_DIR / "keyword_baseline_scores.csv", index=False)

In [28]:
# P3.1–P3.5: Inspect the FastAPI routes and app source file
app_source = (PROJECT_DIR / "app.py").read_text(encoding="utf-8")
print("app.py exists:", (PROJECT_DIR / "app.py").exists())
print("POST /analyze defined:", '@app.post("/analyze"' in app_source)
print("GET /health defined:", '@app.get("/health"' in app_source)
print("Swagger /docs is provided automatically by FastAPI.")
print("API key is read from environment/.env; it is not embedded in app.py.")

app.py exists: True
POST /analyze defined: True
GET /health defined: True
Swagger /docs is provided automatically by FastAPI.
API key is read from environment/.env; it is not embedded in app.py.


In [29]:
# P3.1: Verify the request/response helper without calling the live LLM
from fastapi.testclient import TestClient
import app as app_module

api_test_client = TestClient(app_module.app)
health_response = api_test_client.get("/health")
print("Health status:", health_response.status_code)
print("Health response:", health_response.json())
docs_response = api_test_client.get("/openapi.json")
print("OpenAPI document status:", docs_response.status_code)
print("Available paths:", list(docs_response.json().get("paths", {}).keys()))

2026-11-24 14:13:03,334 INFO request_id=4aa5be09-4792-443d-941d-8fc168f4693b method=GET path=/health status=200 duration_ms=0.0
2026-11-24 14:13:03,344 INFO HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
2026-11-24 14:13:03,372 INFO request_id=6dfb402d-f66a-4b14-a1c0-4f8533d0c6ff method=GET path=/openapi.json status=200 duration_ms=27.23
2026-11-24 14:13:03,381 INFO HTTP Request: GET http://testserver/openapi.json "HTTP/1.1 200 OK"


Health status: 200
Health response: {'status': 'ok', 'service': 'AI Resume Analyzer'}
OpenAPI document status: 200
Available paths: ['/health', '/analyze']


In [30]:
# P3.2: Show the file validation and token-limit settings
print("Allowed file extensions: .pdf and .txt")
print("Maximum upload size (MB):", int(os.getenv("MAX_UPLOAD_MB", "5")))
print("Maximum resume tokens:", core.MAX_RESUME_TOKENS)
print("Scanned PDFs return HTTP 422; unsupported extensions return HTTP 415; oversized files/tokens return HTTP 413.")

Allowed file extensions: .pdf and .txt
Maximum upload size (MB): 5
Maximum resume tokens: 12000
Scanned PDFs return HTTP 422; unsupported extensions return HTTP 415; oversized files/tokens return HTTP 413.


In [31]:
# P3.3–P3.5: Verify secret handling and cost-estimate function
load_dotenv(PROJECT_DIR / ".env")
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("app.py contains a literal API key:", "AIza" in app_source)
sample_usage = {"input_tokens": 1000, "output_tokens": 300, "total_tokens": 1300}
print("Free-tier estimate example:", core.estimate_cost(sample_usage, pricing_tier="free"))
print("Paid-tier reference example:", core.estimate_cost(sample_usage, pricing_tier="paid"))

GEMINI_API_KEY present: False
app.py contains a literal API key: False
Free-tier estimate example: {'pricing_tier': 'free', 'estimated_cost_usd': 0.0, 'paid_standard_reference_usd': 0.00105, 'note': 'Free-tier estimate assumes the request is eligible and within current quota; verify account tier and live pricing.'}
Paid-tier reference example: {'pricing_tier': 'paid', 'estimated_cost_usd': 0.00105, 'paid_standard_reference_usd': 0.00105, 'note': 'Free-tier estimate assumes the request is eligible and within current quota; verify account tier and live pricing.'}


In [32]:
# P3.6–P3.10: Manually test five failure cases without any API call
# These inputs are rejected before the Gemini service is called.
manual_failure_results = []

def record_failure(name, response):
    try:
        message = response.json().get("detail", response.text)
    except Exception:
        message = response.text
    manual_failure_results.append({"failure_case": name, "status_code": response.status_code, "message": str(message)})

# 1. Missing resume source
r = api_test_client.post("/analyze", data={"job_description": "Python Developer role"})
record_failure("Missing resume source", r)
# 2. Wrong file extension
r = api_test_client.post("/analyze", data={"job_description": "Python Developer role"}, files={"resume_file": ("resume.docx", b"not a supported file", "application/octet-stream")})
record_failure("Unsupported file type", r)
# 3. Empty TXT file
r = api_test_client.post("/analyze", data={"job_description": "Python Developer role"}, files={"resume_file": ("empty.txt", b"   ", "text/plain")})
record_failure("Empty resume file", r)
# 4. Scanned/image-only PDF
r = api_test_client.post("/analyze", data={"job_description": "Python Developer role"}, files={"resume_file": ("scanned.pdf", scanned_path.read_bytes(), "application/pdf")})
record_failure("Scanned PDF without selectable text", r)
# 5. Missing/too-short job description
r = api_test_client.post("/analyze", data={"resume_text": "Python developer with SQL", "job_description": "short"})
record_failure("Job description too short", r)

manual_failure_df = pd.DataFrame(manual_failure_results)
display(manual_failure_df)
manual_failure_df.to_csv(OUTPUT_DIR / "api_failure_cases.csv", index=False)

2026-11-24 14:13:13,578 INFO request_id=0eb5c422-ca25-4b9b-bf4f-a24851b850da method=POST path=/analyze status=422 duration_ms=5.25
2026-11-24 14:13:13,596 INFO HTTP Request: POST http://testserver/analyze "HTTP/1.1 422 Unprocessable Content"
2026-11-24 14:13:13,609 INFO request_id=258c89ec-d7a1-41b1-971c-d2597bcaf8a4 method=POST path=/analyze status=415 duration_ms=2.91
2026-11-24 14:13:13,609 INFO HTTP Request: POST http://testserver/analyze "HTTP/1.1 415 Unsupported Media Type"
2026-11-24 14:13:13,663 INFO request_id=ff5ec4fe-e1d2-4b96-ba55-526f14143243 method=POST path=/analyze status=422 duration_ms=20.88
2026-11-24 14:13:13,667 INFO HTTP Request: POST http://testserver/analyze "HTTP/1.1 422 Unprocessable Content"
2026-11-24 14:13:13,696 INFO request_id=e49fc0e7-fe66-4377-aae1-20eca77c21d1 method=POST path=/analyze status=422 duration_ms=10.7
2026-11-24 14:13:13,701 INFO HTTP Request: POST http://testserver/analyze "HTTP/1.1 422 Unprocessable Content"
2026-11-24 14:13:13,715 INFO r

,failure_case,status_code,message
0,Missing resume source,422,Provide exactly one resume source: resume_text...
1,Unsupported file type,415,Unsupported file type. Upload a PDF or TXT res...
2,Empty resume file,422,No readable text was found in the resume.
3,Scanned PDF without selectable text,422,PDF has no selectable text. It may be scanned/...
4,Job description too short,422,"[{'type': 'string_too_short', 'loc': ['body', ..."


In [33]:
# P3.7: Confirm timeout configuration location (do not edit API key values here)
print("The core API uses the Google Gen AI SDK. Its client request timeout can be adjusted with google.genai.types.HttpOptions.")
print("For this training app, keep requests small and use the retry/friendly-error behaviour already in resume_analyzer_core.py.")
print("Do not increase quota by enabling paid billing. If you receive 429, wait and retry later.")

The core API uses the Google Gen AI SDK. Its client request timeout can be adjusted with google.genai.types.HttpOptions.
For this training app, keep requests small and use the retry/friendly-error behaviour already in resume_analyzer_core.py.
Do not increase quota by enabling paid billing. If you receive 429, wait and retry later.


In [34]:
# P3.9: Confirm request-ID middleware adds an ID to health responses
request_id_response = api_test_client.get("/health", headers={"X-Request-ID": "resume-demo-001"})
print("HTTP:", request_id_response.status_code)
print("Returned request ID:", request_id_response.headers.get("X-Request-ID"))

2026-11-24 14:13:20,018 INFO request_id=resume-demo-001 method=GET path=/health status=200 duration_ms=12.43
2026-11-24 14:13:20,022 INFO HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"


HTTP: 200
Returned request ID: resume-demo-001


In [35]:
# P4.1: Run automated TestClient tests. The LLM is mocked; no API key or paid request is used.
pytest_result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=PROJECT_DIR, text=True, capture_output=True)
print(pytest_result.stdout)
if pytest_result.stderr:
    print(pytest_result.stderr)
if pytest_result.returncode != 0:
    raise RuntimeError("Some tests failed. Read the test output above before continuing.")

.............                                                            [100%]
============================== warnings summary ===============================
c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\fitz\__init__.py:17
  c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
13 passed, 1 warning in 2.52s



In [36]:
# P4.5: Check test coverage for the API module
coverage_result = subprocess.run([sys.executable, "-m", "pytest", "--cov=app", "--cov-report=term-missing", "-q"], cwd=PROJECT_DIR, text=True, capture_output=True)
print(coverage_result.stdout)
if coverage_result.stderr:
    print(coverage_result.stderr)
if coverage_result.returncode != 0:
    print("Coverage command failed. Confirm pytest-cov installed with requirements.txt.")

.............                                                            [100%]
============================== warnings summary ===============================
c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\fitz\__init__.py:17
  c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
    warnings.warn(

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
=============================== tests coverage ================================
______________ coverage: platform win32, python 3.12.10-final-0 _______________

Name     Stmts   Miss  Cover   Missing
--------------------------------------
app.py      69      9    87%   76-77, 81, 86, 92, 98-101, 106
--------------------------------------
TOTAL       69      9    87%
13 passed, 1 warning in 6.71s



In [ ]:
# P4.2: Evaluate skill extraction precision and recall against the eight-item golden set
# This compares normalized skill names; it does not claim that every other resume field is perfect.
def normalize_skill(value):
    return " ".join(str(value).lower().strip().split())

metric_rows = []
for gold in golden_set:
    predicted = successful_extractions.get(gold["file"], {})
    expected_skills = {normalize_skill(item) for item in gold["skills"]}
    predicted_skills = {normalize_skill(item) for item in predicted.get("skills", []) if normalize_skill(item) not in {"not provided", ""}}
    true_positive = len(expected_skills & predicted_skills)
    precision = true_positive / len(predicted_skills) if predicted_skills else 0.0
    recall = true_positive / len(expected_skills) if expected_skills else 0.0
    name_correct = normalize_skill(predicted.get("candidate_name", "")) == normalize_skill(gold["candidate_name"])
    metric_rows.append({"resume": gold["file"], "skill_precision": precision, "skill_recall": recall, "candidate_name_correct": name_correct, "expected_skill_count": len(expected_skills), "predicted_skill_count": len(predicted_skills)})
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)
macro_precision = metrics_df["skill_precision"].mean()
macro_recall = metrics_df["skill_recall"].mean()
print("Macro skill precision:", round(macro_precision, 3))
print("Macro skill recall:", round(macro_recall, 3))
print("Candidate-name accuracy:", round(metrics_df["candidate_name_correct"].mean(), 3))
metrics_df.to_csv(OUTPUT_DIR / "extraction_metrics_v1.csv", index=False)

In [ ]:
# P4.3: Report extraction accuracy summary and average score difference
extraction_summary = {
    "resumes_in_golden_set": len(golden_set),
    "successful_extractions": len(successful_extractions),
    "candidate_name_accuracy": round(float(metrics_df["candidate_name_correct"].mean()), 3),
    "macro_skill_precision": round(float(macro_precision), 3),
    "macro_skill_recall": round(float(macro_recall), 3),
    "average_absolute_matching_score_difference": round(float(ranking_df["absolute_score_difference"].mean()), 3) if not ranking_df.empty else None,
}
print(extraction_summary)
(OUTPUT_DIR / "evaluation_summary_v1.json").write_text(json.dumps(extraction_summary, indent=2), encoding="utf-8")

In [39]:
# P4.4: Create an improved prompt (v2) targeting evidence-based extraction
# Improvement: explicitly distinguish an explicit skill from a tool mentioned only in a company/project context.
EXTRACTION_PROMPT_V2 = EXTRACTION_PROMPT_V1 + "\nADDITIONAL EVIDENCE RULES (v2):\n- Include a skill only when the resume explicitly names it as a skill or describes hands-on use of it.\n- Do not add technologies merely because they are common in the candidate's job title or industry.\n- Keep education entries limited to qualifications explicitly written in the resume.\n- Avoid duplicate or overly broad skill labels.\n"
(PROMPT_DIR / "extraction_prompt_v2.txt").write_text(EXTRACTION_PROMPT_V2, encoding="utf-8")
print("Saved improved prompt:", PROMPT_DIR / "extraction_prompt_v2.txt")

Saved improved prompt: d:\New folder\AI_Resume_Analyzer_Project\AI_Resume_Analyzer_Project\prompts\extraction_prompt_v2.txt


In [ ]:
# P4.4: Re-run extraction on the same eight resumes with prompt v2 and compare metrics
# This is another eight Gemini requests. Run only after reviewing v1 metrics and confirming free-tier quota.
extraction_results_v2 = {}
for index, (filename, text) in enumerate(resume_texts.items(), start=1):
    result = core.extract_resume_with_retry(
        text, prompt_template=EXTRACTION_PROMPT_V2, client=client,
        max_attempts=3, pause_seconds=3.0,
    )
    extraction_results_v2[filename] = result
    print(f"v2 {index}/8 {filename}: success={result['success']}, attempts={result['attempts']}")
    time.sleep(6.0)

successful_extractions_v2 = {name: result["data"] for name, result in extraction_results_v2.items() if result["success"]}
(OUTPUT_DIR / "extraction_results_v2.json").write_text(json.dumps(successful_extractions_v2, indent=2), encoding="utf-8")
if len(successful_extractions_v2) != 8:
    raise RuntimeError("Prompt v2 did not successfully process all eight resumes. Review failures before comparing.")

In [ ]:
# P4.4: Compare v1 and v2 skill precision/recall (use actual results; improvement is not guaranteed)
comparison_rows = []
for gold in golden_set:
    pred_v1 = successful_extractions.get(gold["file"], {})
    pred_v2 = successful_extractions_v2.get(gold["file"], {})
    expected = {normalize_skill(s) for s in gold["skills"]}
    def calc_pr(pred):
        predicted = {normalize_skill(s) for s in pred.get("skills", []) if normalize_skill(s) not in {"", "not provided"}}
        tp = len(expected & predicted)
        return (tp / len(predicted) if predicted else 0.0, tp / len(expected) if expected else 0.0)
    p1, r1 = calc_pr(pred_v1)
    p2, r2 = calc_pr(pred_v2)
    comparison_rows.append({"resume": gold["file"], "v1_precision": p1, "v1_recall": r1, "v2_precision": p2, "v2_recall": r2})
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
print("V1 macro precision/recall:", round(comparison_df.v1_precision.mean(), 3), round(comparison_df.v1_recall.mean(), 3))
print("V2 macro precision/recall:", round(comparison_df.v2_precision.mean(), 3), round(comparison_df.v2_recall.mean(), 3))
comparison_df.to_csv(OUTPUT_DIR / "prompt_improvement_comparison.csv", index=False)

In [ ]:
# P4.6: Verify README coverage and the .env.example file
readme_text = (PROJECT_DIR / "README.md").read_text(encoding="utf-8")
readme_checks = {
    "purpose": "Project" in readme_text or "project" in readme_text,
    "architecture/topics": "Part" in readme_text,
    "setup": "Install packages" in readme_text or "Setup" in readme_text,
    "env_example": (PROJECT_DIR / ".env.example").exists(),
    "run_steps": "uvicorn" in readme_text,
    "curl_example": "curl" in readme_text,
    "limitations": "scanned" in readme_text.lower() or "limit" in readme_text.lower(),
    "fairness": "fair" in readme_text.lower() or "human" in readme_text.lower(),
}
print(readme_checks)
assert all(readme_checks.values()), "Review the README sections marked False."

In [46]:
# Final assessment audit: verify the required project files exist
final_required_files = [
    "resume_analyzer.ipynb", "resume_analyzer_core.py", "app.py", "requirements.txt",
    ".env.example", ".gitignore", "README.md",
    "prompts/extraction_prompt_v1.txt", "prompts/matching_prompt_v1.txt",
    "data/golden_set.json", "tests/test_app.py",
]
final_missing = [name for name in final_required_files if not (PROJECT_DIR / name).exists()]
print("Required project files:", len(final_required_files))
print("Missing files:", final_missing)
print("Eight dummy resumes:", len(resume_files))
print("Mandatory checklist: 38 items across Parts 1–4; complete each from trainer's sheet before final submission.")
if final_missing:
    raise FileNotFoundError(final_missing)
print("Project bundle structure check passed. Do not mark the assessment complete until every trainer requirement and output has been reviewed.")

Required project files: 11
Missing files: []
Eight dummy resumes: 8
Mandatory checklist: 38 items across Parts 1–4; complete each from trainer's sheet before final submission.
Project bundle structure check passed. Do not mark the assessment complete until every trainer requirement and output has been reviewed.
